In [1]:
import os
import numpy as np
from PIL import Image
import torch
import torch.utils.data as data


In [2]:
def parse_abs_pose_txt(fpath):
    '''Define how to parse files to get pose labels
        Our pose label format: 
            3 header lines
            list of samples with format: 
                image x y z w p q r
    '''
    ims = []
    f = open(fpath)
    for line in f.readlines()[3::]:    # Skip 3 header lines            
        cur = line.split(' ')
        ims.append(cur[0])
    f.close()
    return ims


In [3]:
from hloc.colmap_from_nvm import read_nvm_model

In [4]:
def quaternion_to_rotation_matrix(qvec):
    qvec = qvec / np.linalg.norm(qvec)
    w, x, y, z = qvec
    R = np.array([
        [1 - 2 * y * y - 2 * z * z, 2 * x * y - 2 * z * w, 2 * x * z + 2 * y * w],
        [2 * x * y + 2 * z * w, 1 - 2 * x * x - 2 * z * z, 2 * y * z - 2 * x * w],
        [2 * x * z - 2 * y * w, 2 * y * z + 2 * x * w, 1 - 2 * x * x - 2 * y * y]])
    return R


def camera_center_to_translation(c, qvec):
    R = quaternion_to_rotation_matrix(qvec)
    return (-1) * np.matmul(R, c)


In [5]:
import argparse
import sqlite3
from tqdm import tqdm
from collections import defaultdict
import numpy as np
from pathlib import Path

from hloc import logger
from hloc.utils.read_write_model import Camera, Image, Point3D, CAMERA_MODEL_NAMES
from hloc.utils.read_write_model import write_model

def read_nvm_model(
        nvm_path, skip_points=True):

    cameras = {}
    nvm_f = open(nvm_path, 'r')
    line = nvm_f.readline()
    while line == '\n' or line.startswith('NVM_V3'):
        line = nvm_f.readline()
    num_images = int(line)
    logger.info(f'Reading {num_images} images...')
    image_data = []
    i = 0
    while i < num_images:
        line = nvm_f.readline()
        if line == '\n':
            continue
        data = line.strip('\n').replace('\t',' ').split(' ')
        image_data.append(data)
        i += 1

    line = nvm_f.readline()
    while line == '\n':
        line = nvm_f.readline()
    num_points = int(line)

    if skip_points:
        logger.info(f'Skipping {num_points} points.')
        num_points = 0
    else:
        logger.info(f'Reading {num_points} points...')
    points3D = {}
    i = 0
    logger.info('Parsing image data...')
    images = {}
    for i, data in enumerate(image_data):
        name, focal, qw, qx, qy, qz, cx, cy, cz, k1, k2, k3 = data
        name = name.replace("jpg", "png")
        qvec = np.array([qw, qx, qy, qz], float)
        c = np.array([cx, cy, cz], float)
        t = camera_center_to_translation(c, qvec)


        
        image = Image(
            id=i,
            qvec=qvec,
            tvec=t,
            camera_id=i,
            name=name,
            xys={},
            point3D_ids={})
        camera_model, width, height, principal_x, principal_y = "SIMPLE_RADIAL_FISHEYE", 1920, 1080, 1920//2, 1080//2
        params = [float(focal), float(principal_x), float(principal_y), -float(k1)]# Note: colmap is inverse https://github.com/colmap/colmap/blob/0478d88a4eb695d7c89a8313a30f0061803522fa/src/colmap/scene/reconstruction.cc#L755
        camera = Camera(
            id=i, model=camera_model,
            width=int(width), height=int(height), params=params)
        cameras[i] = camera
        images[i] = image

    return cameras, images, points3D


In [55]:

scene =  "GreatCourt"

split_test = set(parse_abs_pose_txt(f"datasets/cambridge/{scene}/dataset_test.txt"))
split_train = set(parse_abs_pose_txt(f"datasets/cambridge/{scene}/dataset_train.txt"))


In [56]:
len(split_train) + len(split_test)

2292

In [57]:
cameras, images, points3D = read_nvm_model(f"datasets/cambridge/{scene}/reconstruction.nvm")

[2023/11/10 14:13:31 hloc INFO] Reading 2292 images...
[2023/11/10 14:13:31 hloc INFO] Skipping 205228 points.
[2023/11/10 14:13:31 hloc INFO] Parsing image data...


In [61]:
reconstr_image_names = set([im.name for im in images.values()])

In [62]:
from hloc.extract_features import ImageDataset
all_image_names = set(["/".join(name.split("/")[-2:]) for name in ImageDataset(f"datasets/cambridge/{scene}", {}).names])

[2023/11/10 14:14:54 hloc INFO] Found 2294 images in root datasets/cambridge/GreatCourt.


In [63]:
not_in_reconstr = all_image_names.difference(reconstr_image_names)
if len(not_in_reconstr) > 0:
    print(not_in_reconstr)
    print("Warning, some images missing from reconstruction, make sure to remove them")

{'frame00593.png', 'frame00073.png'}
Warning, some images missing from reconstruction, make sure to remove them


In [64]:
images_test = {i:image for i,image in images.items() if image.name in split_test}
images_train = {i:image for i,image in images.items() if image.name in split_train}

In [65]:
images_train

{0: Image(id=0, qvec=array([ 0.5740694 ,  0.56654934, -0.4194737 ,  0.41654289]), tvec=array([ 16.58572596,   0.77573382, -35.23024546]), camera_id=0, name='seq5/frame00065.png', xys={}, point3D_ids={}),
 1: Image(id=1, qvec=array([ 0.5650592 ,  0.55397166, -0.43389201,  0.43076818]), tvec=array([ 19.02381324,   0.54843849, -34.80177717]), camera_id=1, name='seq5/frame00066.png', xys={}, point3D_ids={}),
 2: Image(id=2, qvec=array([ 0.5910655 ,  0.55117839, -0.42227123,  0.41052523]), tvec=array([ 15.79732555,  -0.86483321, -35.25033151]), camera_id=2, name='seq5/frame00064.png', xys={}, point3D_ids={}),
 3: Image(id=3, qvec=array([ 0.58953524,  0.57417725, -0.39677977,  0.40661345]), tvec=array([ 13.12257086,   0.34472849, -35.12781069]), camera_id=3, name='seq5/frame00063.png', xys={}, point3D_ids={}),
 4: Image(id=4, qvec=array([ 0.59739018,  0.57522018, -0.40098866,  0.38917196]), tvec=array([ 11.64976804,   0.28511658, -35.4541179 ]), camera_id=4, name='seq5/frame00062.png', xys={

In [66]:
colmap_points = {}

output_path = f"datasets/cambridge/{scene}/colmap/test"
os.makedirs(output_path, exist_ok=True)
ext = ".txt"
write_model(cameras, images_test, points3D, str(output_path), ext=ext)

({0: Camera(id=0, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1660.1229248, 960.0, 540.0, 0.0445904086633]),
  1: Camera(id=1, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1660.16845703, 960.0, 540.0, 0.0445475428989]),
  2: Camera(id=2, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1660.16442871, 960.0, 540.0, 0.0444996063897]),
  3: Camera(id=3, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.90942383, 960.0, 540.0, 0.0443355615819]),
  4: Camera(id=4, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.84277344, 960.0, 540.0, 0.0442684917791]),
  5: Camera(id=5, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.76989746, 960.0, 540.0, 0.0441860823614]),
  6: Camera(id=6, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.67712402, 960.0, 540.0, 0.0441067102102]),
  7: Camera(id=7, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.5805

In [67]:
colmap_points = {}

output_path = f"datasets/cambridge/{scene}/colmap/train"
os.makedirs(output_path, exist_ok=True)
ext = ".txt"
write_model(cameras, images_train, points3D, str(output_path), ext=ext)

({0: Camera(id=0, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1660.1229248, 960.0, 540.0, 0.0445904086633]),
  1: Camera(id=1, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1660.16845703, 960.0, 540.0, 0.0445475428989]),
  2: Camera(id=2, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1660.16442871, 960.0, 540.0, 0.0444996063897]),
  3: Camera(id=3, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.90942383, 960.0, 540.0, 0.0443355615819]),
  4: Camera(id=4, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.84277344, 960.0, 540.0, 0.0442684917791]),
  5: Camera(id=5, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.76989746, 960.0, 540.0, 0.0441860823614]),
  6: Camera(id=6, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.67712402, 960.0, 540.0, 0.0441067102102]),
  7: Camera(id=7, model='SIMPLE_RADIAL_FISHEYE', width=1920, height=1080, params=[1659.5805